## Example 1: Basic KD-Tree Construction and Nearest Neighbor Search

In [ ]:
from pykdtree.kdtree import KDTree
import numpy as np

# Example 1: Basic KD-tree construction and nearest neighbor search
print("=== Basic KD-Tree Operations ===")

# Create sample 2D points (simulating geographic coordinates)
np.random.seed(42)
cities = np.random.rand(1000, 2) * 100  # 1000 cities in 100x100 grid

print(f"Created dataset: {cities.shape[0]} points in {cities.shape[1]}D space")
print(f"Data bounds: x=[{cities[:, 0].min():.2f}, {cities[:, 0].max():.2f}], y=[{cities[:, 1].min():.2f}, {cities[:, 1].max():.2f}]")

# Build KD-tree
tree = KDTree(cities)
print(f"✓ Built KD-tree: {type(tree).__name__}")

# Single point nearest neighbor search
query_point = np.array([[50.0, 50.0]])  # Query must be 2D array
k_neighbors = 5

distances, indices = tree.query(query_point, k=k_neighbors)

print(f"\nNearest neighbor search from point {query_point[0]}:")
print(f"✓ Found {len(indices[0])} nearest neighbors")

for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
    neighbor = cities[idx]
    print(f"  {i+1}. Point [{neighbor[0]:.2f}, {neighbor[1]:.2f}] at distance {dist:.3f}")

# Verify distances manually for first neighbor
manual_dist = np.sqrt(np.sum((query_point[0] - cities[indices[0][0]])**2))
print(f"✓ Distance verification: computed={distances[0][0]:.6f}, manual={manual_dist:.6f}")

# Test with different k values
k_values = [1, 3, 10, 50]
print(f"\nTesting different k values:")
for k in k_values:
    dist_k, idx_k = tree.query(query_point, k=k)
    avg_distance = dist_k[0].mean()
    print(f"  k={k}: average distance to {k} nearest = {avg_distance:.3f}")

print("✓ Basic KD-tree operations demonstrated")

## Example 2: Batch Queries and Multi-dimensional Data

In [ ]:
from pykdtree.kdtree import KDTree
import numpy as np

# Example 2: Batch queries and multi-dimensional data
print("=== Batch Queries and Multi-dimensional Data ===")

# Create 3D dataset (simulating RGB color space)
np.random.seed(123)
colors = np.random.rand(5000, 3) * 255  # 5000 colors in RGB space

print(f"Created 3D dataset: {colors.shape[0]} points in RGB color space")
print(f"Color ranges: R=[{colors[:, 0].min():.1f}, {colors[:, 0].max():.1f}], "
      f"G=[{colors[:, 1].min():.1f}, {colors[:, 1].max():.1f}], "
      f"B=[{colors[:, 2].min():.1f}, {colors[:, 2].max():.1f}]")

# Build 3D KD-tree
color_tree = KDTree(colors)
print(f"✓ Built 3D KD-tree")

# Batch query with multiple target colors
target_colors = np.array([
    [255.0, 0.0, 0.0],    # Pure red
    [0.0, 255.0, 0.0],    # Pure green  
    [0.0, 0.0, 255.0],    # Pure blue
    [128.0, 128.0, 128.0], # Gray
    [255.0, 255.0, 0.0]   # Yellow
])

print(f"\nBatch query for {len(target_colors)} target colors:")

# Perform batch nearest neighbor search
batch_distances, batch_indices = color_tree.query(target_colors, k=3)

color_names = ['Red', 'Green', 'Blue', 'Gray', 'Yellow']
for i, (color_name, target) in enumerate(zip(color_names, target_colors)):
    print(f"\n{color_name} target {target}:")
    for j, (dist, idx) in enumerate(zip(batch_distances[i], batch_indices[i])):
        closest_color = colors[idx]
        print(f"  {j+1}. RGB({closest_color[0]:.1f}, {closest_color[1]:.1f}, {closest_color[2]:.1f}) "
              f"distance: {dist:.2f}")

# Test higher dimensional data
print(f"\nHigh-dimensional scaling test:")
dimensions = [2, 3, 5, 10, 20]

for dim in dimensions:
    # Create random dataset in d-dimensional space
    data_nd = np.random.rand(1000, dim) * 100
    query_nd = np.random.rand(10, dim) * 100  # 10 query points
    
    # Build tree and perform queries
    tree_nd = KDTree(data_nd)
    distances_nd, indices_nd = tree_nd.query(query_nd, k=5)
    
    # Calculate average distances
    avg_dist = distances_nd.mean()
    min_dist = distances_nd.min()
    max_dist = distances_nd.max()
    
    print(f"  {dim}D: avg_dist={avg_dist:.2f}, min_dist={min_dist:.2f}, max_dist={max_dist:.2f}")

# Demonstrate curse of dimensionality
print(f"\nCurse of dimensionality demonstration:")
base_size = 1000
for dim in [2, 5, 10, 20, 50]:
    data_curse = np.random.rand(base_size, dim)
    query_curse = np.random.rand(1, dim)
    
    tree_curse = KDTree(data_curse)
    dist_curse, _ = tree_curse.query(query_curse, k=10)
    
    nearest_dist = dist_curse[0][0]  # Distance to nearest neighbor
    farthest_dist = dist_curse[0][-1]  # Distance to 10th nearest
    ratio = farthest_dist / nearest_dist
    
    print(f"  {dim}D: nearest={nearest_dist:.4f}, 10th={farthest_dist:.4f}, ratio={ratio:.2f}")

print("✓ Batch queries and multi-dimensional analysis completed")

## Example 3: Performance Analysis and Radius-based Searches

In [ ]:
from pykdtree.kdtree import KDTree
import numpy as np
import time

# Example 3: Performance analysis and radius-based searches
print("=== Performance Analysis and Radius-based Searches ===")

# Performance benchmark with different dataset sizes
dataset_sizes = [1000, 5000, 10000, 50000]
query_sizes = [1, 10, 100]

print("Performance benchmarks:")
print("Dataset Size | Queries | Build Time | Query Time | Time/Query")
print("-" * 60)

for data_size in dataset_sizes:
    # Create dataset
    np.random.seed(42)
    data = np.random.rand(data_size, 3) * 1000  # 3D points
    
    # Build tree and measure time
    start_time = time.time()
    tree = KDTree(data)
    build_time = time.time() - start_time
    
    for query_size in query_sizes:
        queries = np.random.rand(query_size, 3) * 1000
        
        # Measure query time
        start_time = time.time()
        distances, indices = tree.query(queries, k=10)
        query_time = time.time() - start_time
        
        time_per_query = query_time / query_size * 1000  # ms per query
        
        print(f"{data_size:>11} | {query_size:>7} | {build_time:>9.4f}s | {query_time:>9.4f}s | {time_per_query:>8.2f}ms")

# Simulate radius-based search using k-nearest neighbors
print(f"\nRadius-based search simulation:")

# Create 2D dataset for spatial analysis
np.random.seed(456)
spatial_points = np.random.rand(2000, 2) * 100
tree_spatial = KDTree(spatial_points)

# Define search center and radius
search_center = np.array([[50.0, 50.0]])
search_radius = 10.0

print(f"Search center: {search_center[0]}")
print(f"Search radius: {search_radius}")

# Strategy: Get many neighbors and filter by distance
max_neighbors = 200  # Get up to 200 neighbors
distances, indices = tree_spatial.query(search_center, k=max_neighbors)

# Filter points within radius
within_radius = distances[0] <= search_radius
radius_distances = distances[0][within_radius]
radius_indices = indices[0][within_radius]

print(f"✓ Points within radius: {len(radius_indices)}")
print(f"✓ Distance range: {radius_distances.min():.2f} to {radius_distances.max():.2f}")

# Show first few points within radius
print("Points within radius:")
for i, (dist, idx) in enumerate(zip(radius_distances[:5], radius_indices[:5])):
    point = spatial_points[idx]
    print(f"  {i+1}. Point [{point[0]:.2f}, {point[1]:.2f}] at distance {dist:.2f}")

# Test different radius values
radius_values = [5.0, 10.0, 15.0, 20.0, 25.0]
print(f"\nRadius analysis:")

for radius in radius_values:
    # Use sufficient k to capture all points within radius
    k_large = min(500, len(spatial_points))
    dist_all, idx_all = tree_spatial.query(search_center, k=k_large)
    
    # Filter by radius
    within_r = dist_all[0] <= radius
    count_within = np.sum(within_r)
    
    if count_within > 0:
        avg_dist = dist_all[0][within_r].mean()
        print(f"  Radius {radius:>4.1f}: {count_within:>3d} points, avg_distance={avg_dist:.2f}")
    else:
        print(f"  Radius {radius:>4.1f}: {count_within:>3d} points")

# Memory and scalability analysis
print(f"\nMemory and scalability analysis:")

for size in [1000, 10000, 100000]:
    large_data = np.random.rand(size, 3)
    
    # Time tree construction
    start_time = time.time()
    large_tree = KDTree(large_data)
    construction_time = time.time() - start_time
    
    # Estimate memory (approximate)
    points_memory = large_data.nbytes
    overhead_estimate = size * 32  # Rough estimate of tree overhead in bytes
    
    print(f"  {size:>6d} points: build_time={construction_time:.4f}s, "
          f"data={points_memory//1024}KB, est_overhead={overhead_estimate//1024}KB")

print("✓ Performance analysis and radius-based searches completed")

## Example 4: Geospatial Applications and Distance Metrics

In [ ]:
from pykdtree.kdtree import KDTree
import numpy as np

# Example 4: Geospatial applications and distance metrics
print("=== Geospatial Applications and Distance Metrics ===")

# Create realistic geographic dataset (latitude, longitude pairs)
np.random.seed(789)

# Simulate city locations around San Francisco Bay Area
# Latitude range: ~37.3 to 37.9, Longitude range: ~-122.5 to -121.9
lat_range = (37.3, 37.9)
lon_range = (-122.5, -121.9)

num_locations = 1000
latitudes = np.random.uniform(lat_range[0], lat_range[1], num_locations)
longitudes = np.random.uniform(lon_range[0], lon_range[1], num_locations)

# Combine into coordinate pairs
coordinates = np.column_stack((latitudes, longitudes))

print(f"Generated {num_locations} locations in Bay Area")
print(f"Latitude range: [{lat_range[0]}, {lat_range[1]}]")
print(f"Longitude range: [{lon_range[0]}, {lon_range[1]}]")

# Build KD-tree for geographic data
geo_tree = KDTree(coordinates)
print(f"✓ Built geographic KD-tree")

# Define some landmark locations for queries
landmarks = {
    'San Francisco': [37.7749, -122.4194],
    'San Jose': [37.3382, -121.8863], 
    'Oakland': [37.8044, -122.2711],
    'Palo Alto': [37.4419, -122.1430],
    'Berkeley': [37.8715, -122.2730]
}

print(f"\nFinding nearest locations to landmarks:")

for name, coords in landmarks.items():
    query_point = np.array([coords])  # Must be 2D array
    distances, indices = geo_tree.query(query_point, k=5)
    
    print(f"\n{name} ({coords[0]:.4f}, {coords[1]:.4f}):")
    for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
        nearest_coord = coordinates[idx]
        
        # Convert distance from degrees to approximate kilometers
        # Simple approximation: 1 degree ≈ 111 km
        dist_km = dist * 111
        
        print(f"  {i+1}. Location ({nearest_coord[0]:.4f}, {nearest_coord[1]:.4f}) "
              f"distance: {dist:.6f}° ({dist_km:.2f} km)")

# Demonstrate distance calculations in different contexts
print(f"\nDistance metric analysis:")

# Create test points for distance calculation
test_points = np.array([
    [37.7749, -122.4194],  # SF
    [37.3382, -121.8863],  # San Jose
])

# Calculate Euclidean distance in coordinate space
euclidean_dist = np.sqrt(np.sum((test_points[0] - test_points[1])**2))
print(f"SF to San Jose:")
print(f"  Euclidean distance in degrees: {euclidean_dist:.6f}")
print(f"  Approximate distance in km: {euclidean_dist * 111:.2f}")

# Haversine formula for more accurate great-circle distance
def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate great-circle distance using Haversine formula"""
    R = 6371  # Earth's radius in km
    
    lat1_rad = np.radians(lat1)
    lat2_rad = np.radians(lat2)
    delta_lat = np.radians(lat2 - lat1)
    delta_lon = np.radians(lon2 - lon1)
    
    a = (np.sin(delta_lat/2)**2 + 
         np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(delta_lon/2)**2)
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    
    return R * c

haversine_dist = haversine_distance(test_points[0][0], test_points[0][1], 
                                   test_points[1][0], test_points[1][1])
print(f"  Haversine distance (accurate): {haversine_dist:.2f} km")

# Create density analysis around landmarks
print(f"\nLocation density analysis:")

for name, coords in list(landmarks.items())[:3]:  # First 3 landmarks
    query_point = np.array([coords])
    
    # Count locations within different radii
    radii_degrees = [0.01, 0.05, 0.1, 0.2]  # In degrees
    
    print(f"\n{name} density analysis:")
    for radius in radii_degrees:
        # Get many neighbors and filter by radius
        k_large = min(200, num_locations)
        distances, indices = geo_tree.query(query_point, k=k_large)
        
        within_radius = np.sum(distances[0] <= radius)
        radius_km = radius * 111  # Convert to approximate km
        
        if within_radius > 0:
            density = within_radius / (np.pi * radius_km**2)  # locations per km²
            print(f"  Within {radius_km:>5.1f} km: {within_radius:>3d} locations "
                  f"(density: {density:.2f} loc/km²)")

# Simulate route optimization (traveling salesman preview)
print(f"\nRoute optimization preview:")

# Select 5 random locations for a simple route
route_indices = np.random.choice(num_locations, 5, replace=False)
route_locations = coordinates[route_indices]

print("Selected locations for route:")
for i, loc in enumerate(route_locations):
    print(f"  Stop {i+1}: ({loc[0]:.4f}, {loc[1]:.4f})")

# Calculate total distance for this route
total_distance = 0
for i in range(len(route_locations)):
    start = route_locations[i]
    end = route_locations[(i + 1) % len(route_locations)]  # Return to start
    
    # Find nearest neighbor distance (approximation)
    query = np.array([end])
    tree_temp = KDTree(np.array([start]))
    dist, _ = tree_temp.query(query, k=1)
    
    segment_distance = dist[0] * 111  # Convert to km
    total_distance += segment_distance
    
    print(f"  Segment {i+1}→{(i+1)%len(route_locations)+1}: {segment_distance:.2f} km")

print(f"✓ Total route distance: {total_distance:.2f} km")

print("✓ Geospatial applications and distance metrics demonstrated")

## Example 5: Machine Learning Applications and Clustering

In [ ]:
from pykdtree.kdtree import KDTree
import numpy as np

# Example 5: Machine learning applications and clustering
print("=== Machine Learning Applications and Clustering ===")

# Generate synthetic dataset with clusters for ML applications
np.random.seed(100)

# Create 4 distinct clusters in 2D space
cluster_centers = np.array([
    [20, 20],   # Cluster 1
    [80, 20],   # Cluster 2  
    [20, 80],   # Cluster 3
    [80, 80]    # Cluster 4
])

points_per_cluster = 250
std_dev = 8  # Cluster spread

# Generate points around each center
all_points = []
cluster_labels = []

for i, center in enumerate(cluster_centers):
    cluster_points = np.random.normal(center, std_dev, (points_per_cluster, 2))
    all_points.extend(cluster_points)
    cluster_labels.extend([i] * points_per_cluster)

dataset = np.array(all_points)
true_labels = np.array(cluster_labels)

print(f"Generated ML dataset: {dataset.shape[0]} points in {len(cluster_centers)} clusters")

# Build KD-tree for the dataset
ml_tree = KDTree(dataset)
print(f"✓ Built KD-tree for ML dataset")

# k-NN Classification Demo
print(f"\nk-NN Classification Demo:")

test_points = np.array([
    [25, 25],   # Should be cluster 0
    [75, 25],   # Should be cluster 1
    [25, 75],   # Should be cluster 2
    [75, 75]    # Should be cluster 3
])

k_values = [1, 3, 5, 10]

for i, test_point in enumerate(test_points):
    print(f"\nTest point {i+1}: [{test_point[0]}, {test_point[1]}]")
    
    for k in k_values:
        # Find k nearest neighbors - must use numpy array
        query = np.array([test_point])
        distances, indices = ml_tree.query(query, k=k)
        
        # Get labels of nearest neighbors
        neighbor_labels = true_labels[indices[0]]
        
        # Vote for most common label
        unique_labels, counts = np.unique(neighbor_labels, return_counts=True)
        predicted_label = unique_labels[np.argmax(counts)]
        confidence = np.max(counts) / k
        
        print(f"  k={k:>2}: Cluster{predicted_label}, conf={confidence:.2f}")

# Clustering quality assessment
print(f"\nClustering quality assessment:")

k_cluster = 10
cluster_purities = []

for cluster_id in range(len(cluster_centers)):
    cluster_mask = true_labels == cluster_id
    cluster_points = dataset[cluster_mask]
    cluster_indices = np.where(cluster_mask)[0]
    
    purities = []
    sample_size = min(50, len(cluster_points))
    sample_indices = np.random.choice(len(cluster_points), sample_size, replace=False)
    
    for sample_idx in sample_indices:
        point_idx = cluster_indices[sample_idx]
        query_point = dataset[point_idx:point_idx+1]
        
        distances, neighbor_indices = ml_tree.query(query_point, k=k_cluster+1)
        neighbor_indices = neighbor_indices[0][1:]
        neighbor_labels = true_labels[neighbor_indices]
        
        same_cluster = np.sum(neighbor_labels == cluster_id)
        purity = same_cluster / len(neighbor_labels)
        purities.append(purity)
    
    avg_purity = np.mean(purities)
    cluster_purities.append(avg_purity)
    print(f"  Cluster {cluster_id}: purity = {avg_purity:.3f}")

overall_purity = np.mean(cluster_purities)
print(f"✓ Overall clustering purity: {overall_purity:.3f}")

print("✓ Machine learning applications completed")